In [6]:
# Imports & NLP Code
import pandas as pd
from transformers import pipeline
import pandas as pd
import requests
from tqdm import tqdm

def process_text(text):    
    tdf1 = _tag_genes(text)
    tdf2 = _tag_chemicals(text)
    tdf3 = _tag_diseases(text)
    df = pd.concat([tdf1,tdf2,tdf3])
    df = _drop_unknowns(df)
    df = normalize(df)
    return df

def batch(corpus):
    df = pd.DataFrame()
    for entry in corpus:
        tdf1 = _tag_genes(entry)
        tdf2 = _tag_chemicals(entry)
        tdf3 = _tag_diseases(entry)
        df = pd.concat([df,tdf1,tdf2,tdf3])
    df = _drop_unknowns(df)
    df['start'] = df['start'].astype(int)
    df['end'] = df['end'].astype(int)
    df = normalize(df)
    return df

def normalize(result):
    result['concept_match_type'] = None
    result['concept_id'] = None
    result['concept_label'] = None
    for index, row in tqdm(result.iterrows()):
        if row['entity_group'] == 'GENETIC':
            norm_result = _normalize_gene(row['word'])
        if row['entity_group'] == 'CHEMICAL':
            norm_result = _normalize_therapy(row['word'])
        if row['entity_group'] == 'DISEASE':
            norm_result = _normalize_disease(row['word'])
        result.at[index, 'concept_match_type'] = norm_result[0]
        result.at[index, 'concept_id'] = norm_result[1]
        result.at[index, 'concept_label'] = norm_result[2]
    return result


# TODO: Implement cached queries to improve normalization time -- Brian Walsh/ Kori / James
def _normalize_gene(word):
    r = requests.get(f'https://normalize.cancervariants.org/gene/normalize?q={word}')
    response = r.json()
    if response['match_type'] != 0:
        match_type = response['match_type']
        concept_id = response['gene']['id']
        label = response['gene']['name']
    else:
        match_type = response['match_type']
        concept_id = None
        label = None

    return [match_type, concept_id, label]

def _normalize_disease(word):
    r = requests.get(f'https://normalize.cancervariants.org/disease/normalize?q={word}')
    response = r.json()
    if response['match_type'] != 0:
        match_type = response['match_type']
        concept_id = response['disease']['id']
        label = response['disease']['name']
    else:
        match_type = response['match_type']
        concept_id = None
        label = None
    
    return [match_type, concept_id, label]

def _normalize_therapy(word):
    r = requests.get(f'https://normalize.cancervariants.org/therapy/normalize?q={word}&infer_namespace=true')
    response = r.json()
    if response['match_type'] != 0:
        match_type = response['match_type']
        concept_id = response['therapy']['id']
        label = response['therapy']['name']
    else:
        match_type = response['match_type']
        concept_id = None
        label = None

    return [match_type, concept_id, label]


def _drop_unknowns(result):
    try:
        dropped = result[result['entity_group']!='0'].reset_index(drop=True)
    except:
        dropped = result
    return dropped

def _tag_genes(text):
    _pipe_gene = pipeline("token-classification", model="alvaroalon2/biobert_genetic_ner",aggregation_strategy="first")
    gene_results = _pipe_gene(text)
    df = _drop_unknowns(pd.DataFrame(gene_results))
    df['original_text'] = text
    return df

def _tag_chemicals(text):
    _pipe_chemical = pipeline("token-classification", model="alvaroalon2/biobert_chemical_ner", aggregation_strategy="first")
    chem_results = _pipe_chemical(text)
    df = _drop_unknowns(pd.DataFrame(chem_results))
    df['original_text'] = text
    return df

def _tag_diseases(text):
    _pipe_disease = pipeline("token-classification", model="alvaroalon2/biobert_diseases_ner", aggregation_strategy="first")
    disease_results = _pipe_disease(text)
    df = _drop_unknowns(pd.DataFrame(disease_results))
    df['original_text'] = text
    return df

In [2]:
# Download link
# https://www.fda.gov/about-fda/oncology-center-excellence/pediatric-oncology-drug-approvals#:~:text=Downloadable%20file%20for%3A-,Pediatric%20Approvals%20Additional%20Information,-Search%3A

In [3]:
df = pd.read_excel('pediatric_approvals_additional_information_june_01_2025.xlsx')

In [4]:
df.head()

,DRUGS APPROVED FOR PEDIATRIC CANCERS [BRAND NAME] 1,PHARMACOLOGIC CLASS 2,DRUG TARGET(S) 3,INDICATION 4,PEDIATRIC [AGE RANGE],PEDIATRIC APPROVAL DATE 5,PEDIATRIC APPROVAL (ORIGINAL OR SUPPLEMENTAL APPLICATION),CLINICAL TRIALS 6,OTHER INFORMATION,Unnamed: 9,Unnamed: 10
0,Belzutifan [Welireg],hypoxia-inducible factor inhibitor,EPAS1 (HIF2A),Treatment of adult and pediatric patients 12 y...,12 years and older,2025-05-14,Supplemental [Initial U.S. Approval: 08-13-2021],NCT04924075,NaN,NaN,NaN
1,Nivolumab [Opdivo],programmed death receptor-1 (PD-1)-blocking an...,PDCD1 (PD-1),"As a single agent, for the treatment of adult ...",12 years and older,2025-04-08,Supplemental [Initial U.S. Approval: 12-22-2014],"NCT04008030, NCT02060188","On July 31, 2017, the FDA granted accelerated ...",NaN,NaN
2,Nivolumab [Opdivo],programmed death receptor-1 (PD-1)-blocking an...,PDCD1 (PD-1),"In combination with ipilimumab, for the treat...",12 years and older,2025-04-08,Supplemental [Initial U.S. Approval: 12-22-2014],"NCT04008030, NCT02060188","On July 10, 2018, the FDA granted accelerated ...",NaN,NaN
3,Ipilimumab [Yervoy],human cytotoxic T-lymphocyte antigen 4 (CTLA-4...,CTLA4,"In combination with nivolumab, for the treatme...",12 years and older,2025-04-08,Supplemental [Initial U.S. Approval: 03-25-2011],"NCT04008030, NCT02060188","On July 10, 2018, the FDA granted accelerated...",NaN,NaN
4,Cabozantinib [Cabometyx],kinase inhibitor,"MET, FLT1 (VEGFR1) KDR (VEGFR2), FLT4 (VEGFR3)...",Treatment of adult and pediatric patients 12 y...,12 years and older,2025-03-26,Supplemental / [Initial U.S. Approval: 11-29-2...,NCT03375320,NaN,NaN,NaN


In [7]:
df = df.rename(columns={'INDICATION 4       ':'INDICATIONS'})
results = batch(df['INDICATIONS'])

Device set to use mps:0
/Users/mjc014/.pyenv/versions/3.13.2/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Device set to use mps:0
/Users/mjc014/.pyenv/versions/3.13.2/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Device set to use mps:0
/Users/mjc014/.pyenv/versions/3.13.2/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Device set to use mps:0
/Users/mjc014/.pyenv/versions/3.13.2/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: Futu

In [12]:
results['FDA Brand Label'] = None
for idx,row in results.iterrows():
    tdf = df[df['INDICATIONS']==row['original_text']].reset_index(drop=True)
    results.at[idx, 'FDA Brand Label'] = tdf['DRUGS APPROVED FOR PEDIATRIC CANCERS [BRAND NAME] 1'][0]

results



,original_text,entity_group,score,word,start,end,concept_match_type,concept_id,concept_label,FDA Brand Label
0,Treatment of adult and pediatric patients 12 y...,DISEASE,0.999993,pheochromocytoma,112,128,80,normalize.disease.mondo:0008233,pheochromocytoma,Belzutifan [Welireg]
1,Treatment of adult and pediatric patients 12 y...,DISEASE,0.999990,paraganglioma,132,145,80,normalize.disease.ncit:C3308,Paraganglioma,Belzutifan [Welireg]
2,Treatment of adult and pediatric patients 12 y...,DISEASE,0.999990,PPGL,147,151,0,None,None,Belzutifan [Welireg]
3,"As a single agent, for the treatment of adult ...",CHEMICAL,0.999998,fluoropyrimidine,251,267,80,normalize.therapy.ncit:C94728,Fluoropyrimidine,Nivolumab [Opdivo]
4,"As a single agent, for the treatment of adult ...",CHEMICAL,0.999998,oxaliplatin,269,280,80,normalize.therapy.rxcui:32592,oxaliplatin,Nivolumab [Opdivo]
...,...,...,...,...,...,...,...,...,...,...
261,Treatment of adult and pediatric patients with...,DISEASE,0.999992,acute lymphoblastic leukemia,47,75,80,normalize.disease.ncit:C3167,Acute Lymphoblastic Leukemia,Mercaptopurine \n[Purinethol tablet]
262,Treatment of adult and pediatric patients with...,DISEASE,0.999993,ALL,77,80,60,normalize.disease.ncit:C3167,Acute Lymphoblastic Leukemia,Mercaptopurine \n[Purinethol tablet]
263,Remission induction and remission consolidatio...,DISEASE,0.999991,acute nonlymphocytic leukemias,61,91,0,None,None,Thioguanine [Tabloid]
264,Acute leukemia; MULTIPLE (in combination),CHEMICAL,0.961778,MULTIPLE,16,24,0,None,None,Vincristine Sulfate


In [13]:
tdf = results[results['concept_match_type']!=0].reset_index(drop=True)
tdf[tdf['entity_group']=='DISEASE']['FDA Brand Label'].nunique()

56

In [16]:
df = df.rename(columns={'DRUGS APPROVED FOR PEDIATRIC CANCERS [BRAND NAME] 1': 'FDA Brand Label'})
df

,FDA Brand Label,PHARMACOLOGIC CLASS 2,DRUG TARGET(S) 3,INDICATIONS,PEDIATRIC [AGE RANGE],PEDIATRIC APPROVAL DATE 5,PEDIATRIC APPROVAL (ORIGINAL OR SUPPLEMENTAL APPLICATION),CLINICAL TRIALS 6,OTHER INFORMATION,Unnamed: 9,Unnamed: 10
0,Belzutifan [Welireg],hypoxia-inducible factor inhibitor,EPAS1 (HIF2A),Treatment of adult and pediatric patients 12 y...,12 years and older,2025-05-14,Supplemental [Initial U.S. Approval: 08-13-2021],NCT04924075,NaN,NaN,NaN
1,Nivolumab [Opdivo],programmed death receptor-1 (PD-1)-blocking an...,PDCD1 (PD-1),"As a single agent, for the treatment of adult ...",12 years and older,2025-04-08,Supplemental [Initial U.S. Approval: 12-22-2014],"NCT04008030, NCT02060188","On July 31, 2017, the FDA granted accelerated ...",NaN,NaN
2,Nivolumab [Opdivo],programmed death receptor-1 (PD-1)-blocking an...,PDCD1 (PD-1),"In combination with ipilimumab, for the treat...",12 years and older,2025-04-08,Supplemental [Initial U.S. Approval: 12-22-2014],"NCT04008030, NCT02060188","On July 10, 2018, the FDA granted accelerated ...",NaN,NaN
3,Ipilimumab [Yervoy],human cytotoxic T-lymphocyte antigen 4 (CTLA-4...,CTLA4,"In combination with nivolumab, for the treatme...",12 years and older,2025-04-08,Supplemental [Initial U.S. Approval: 03-25-2011],"NCT04008030, NCT02060188","On July 10, 2018, the FDA granted accelerated...",NaN,NaN
4,Cabozantinib [Cabometyx],kinase inhibitor,"MET, FLT1 (VEGFR1) KDR (VEGFR2), FLT4 (VEGFR3)...",Treatment of adult and pediatric patients 12 y...,12 years and older,2025-03-26,Supplemental / [Initial U.S. Approval: 11-29-2...,NCT03375320,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
93,Methotrexate \n,folate analog metabolic inhibitor,DHFR,Treatment of adults and pediatric patients wit...,Pediatric,1953-12-07,Original,NaN,NaN,NaN,NaN
94,Methotrexate \n,folate analog metabolic inhibitor,DHFR,Treatment of adults and pediatric patients wit...,Pediatric,1953-12-07,Original,NaN,NaN,NaN,NaN
95,Mercaptopurine \n[Purinethol tablet],nucleoside metabolic inhibitor,NaN,Treatment of adult and pediatric patients with...,Pediatric,1953-09-11,Original,NaN,"Mercaptopurine (Purixan), oral suspension: 04-...",NaN,NaN
96,Thioguanine [Tabloid],antimetabolite,NaN,Remission induction and remission consolidatio...,NaN,NaT,[Initial U.S. Approval: 01-18-1966],NaN,NaN,NaN,NaN


In [ ]:
# import inflect

# inflector = inflect.engine()

# def singularize(word):
#     return inflector.singular_noun(word) or word

# results['inflect_test'] = results['word'].apply(singularize)
# results[['word','inflect_test']][0:50]

In [14]:
condensed_results = tdf.groupby('FDA Brand Label').apply(
    lambda group: pd.Series({
        'GENETIC_LABELS': ' | '.join(group.loc[group['entity_group'] == 'GENETIC', 'concept_label'].unique()),
        'DISEASE_LABELS': ' | '.join(group.loc[group['entity_group'] == 'DISEASE', 'concept_label'].unique())
    })
).reset_index()
condensed_results

/var/folders/5t/sfw5tjx56m10xb861_pd3wfm0000gq/T/ipykernel_50247/3014900754.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  condensed_results = tdf.groupby('FDA Brand Label').apply(


,FDA Brand Label,GENETIC_LABELS,DISEASE_LABELS
0,Arsenic trioxide [Trisenox],PML,Acute Promyelocytic Leukemia with t(15;17)(q24...
1,Asparaginase Erwinia Chrysanthemi [Erwinaze] [...,,Acute Lymphoblastic Leukemia | Hypersensitivity
2,Asparaginase Erwinia chrysanthemi (recombinan...,,Acute Lymphoblastic Leukemia | Lymphoblastic L...
3,Asparaginase [Elspar],,Acute Lymphoblastic Leukemia
4,Atezolizumab [Tecentriq],,Alveolar Soft Part Sarcoma
5,Avelumab [Bavencio],,Merkel Cell Carcinoma
6,Azacitidine [Vidaza],,Juvenile Myelomonocytic Leukemia
7,Belzutifan [Welireg],,pheochromocytoma | Paraganglioma
8,Blinatumomab [Blincyto],CD19,Acute Lymphoblastic Leukemia
9,Bosutinib [Bosulif],,"Chronic Myeloid Leukemia, BCR-ABL1 Positive"


In [17]:
merged_df = pd.merge(df, condensed_results, on='FDA Brand Label', how='left')
# merged_df.to_excel('tagged-dataset-joined.xlsx')
merged_df

,FDA Brand Label,PHARMACOLOGIC CLASS 2,DRUG TARGET(S) 3,INDICATIONS,PEDIATRIC [AGE RANGE],PEDIATRIC APPROVAL DATE 5,PEDIATRIC APPROVAL (ORIGINAL OR SUPPLEMENTAL APPLICATION),CLINICAL TRIALS 6,OTHER INFORMATION,Unnamed: 9,Unnamed: 10,GENETIC_LABELS,DISEASE_LABELS
0,Belzutifan [Welireg],hypoxia-inducible factor inhibitor,EPAS1 (HIF2A),Treatment of adult and pediatric patients 12 y...,12 years and older,2025-05-14,Supplemental [Initial U.S. Approval: 08-13-2021],NCT04924075,NaN,NaN,NaN,,pheochromocytoma | Paraganglioma
1,Nivolumab [Opdivo],programmed death receptor-1 (PD-1)-blocking an...,PDCD1 (PD-1),"As a single agent, for the treatment of adult ...",12 years and older,2025-04-08,Supplemental [Initial U.S. Approval: 12-22-2014],"NCT04008030, NCT02060188","On July 31, 2017, the FDA granted accelerated ...",NaN,NaN,,Malignant Colorectal Neoplasm | Colorectal Car...
2,Nivolumab [Opdivo],programmed death receptor-1 (PD-1)-blocking an...,PDCD1 (PD-1),"In combination with ipilimumab, for the treat...",12 years and older,2025-04-08,Supplemental [Initial U.S. Approval: 12-22-2014],"NCT04008030, NCT02060188","On July 10, 2018, the FDA granted accelerated ...",NaN,NaN,,Malignant Colorectal Neoplasm | Colorectal Car...
3,Ipilimumab [Yervoy],human cytotoxic T-lymphocyte antigen 4 (CTLA-4...,CTLA4,"In combination with nivolumab, for the treatme...",12 years and older,2025-04-08,Supplemental [Initial U.S. Approval: 03-25-2011],"NCT04008030, NCT02060188","On July 10, 2018, the FDA granted accelerated...",NaN,NaN,,Malignant Colorectal Neoplasm | Colorectal Car...
4,Cabozantinib [Cabometyx],kinase inhibitor,"MET, FLT1 (VEGFR1) KDR (VEGFR2), FLT4 (VEGFR3)...",Treatment of adult and pediatric patients 12 y...,12 years and older,2025-03-26,Supplemental / [Initial U.S. Approval: 11-29-2...,NCT03375320,NaN,NaN,NaN,KDR,Primitive Neuroectodermal Tumor | Differentiat...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,Methotrexate \n,folate analog metabolic inhibitor,DHFR,Treatment of adults and pediatric patients wit...,Pediatric,1953-12-07,Original,NaN,NaN,NaN,NaN,,Meningeal Leukemia | Acute Lymphoblastic Leuke...
94,Methotrexate \n,folate analog metabolic inhibitor,DHFR,Treatment of adults and pediatric patients wit...,Pediatric,1953-12-07,Original,NaN,NaN,NaN,NaN,,Meningeal Leukemia | Acute Lymphoblastic Leuke...
95,Mercaptopurine \n[Purinethol tablet],nucleoside metabolic inhibitor,NaN,Treatment of adult and pediatric patients with...,Pediatric,1953-09-11,Original,NaN,"Mercaptopurine (Purixan), oral suspension: 04-...",NaN,NaN,,Acute Lymphoblastic Leukemia
96,Thioguanine [Tabloid],antimetabolite,NaN,Remission induction and remission consolidatio...,NaN,NaT,[Initial U.S. Approval: 01-18-1966],NaN,NaN,NaN,NaN,NaN,NaN
